# Import

In [ ]:
!pip install git+https://github.com/huggingface/transformers@82a06db03535c49aa987719ed0746a76093b1ec4

  Cloning https://github.com/huggingface/transformers (to revision 82a06db03535c49aa987719ed0746a76093b1ec4) to /tmp/pip-req-build-d_v05vzz
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-d_v05vzz
  Running command git rev-parse -q --verify 'sha^82a06db03535c49aa987719ed0746a76093b1ec4'
  Running command git fetch -q https://github.com/huggingface/transformers 82a06db03535c49aa987719ed0746a76093b1ec4
  Running command git checkout -q 82a06db03535c49aa987719ed0746a76093b1ec4
  Resolved https://github.com/huggingface/transformers to commit 82a06db03535c49aa987719ed0746a76093b1ec4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.3 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-4.57.1.dev0-py3-none-any.whl size=12023111 sha256=4adec767dcfff46c1ccf340

# HunYuanVLForConditionalGeneration

In [ ]:
import inspect
from transformers import HunYuanVLForConditionalGeneration

print(inspect.getsource(HunYuanVLForConditionalGeneration))

class HunYuanVLForConditionalGeneration(HunYuanVLPreTrainedModel, GenerationMixin):
    _tied_weights_keys = ["lm_head.weight"]
    config: HunYuanVLConfig

    def __init__(self, config: HunYuanVLConfig):
        super().__init__(config)
        self.model = HunYuanVLModel(config)
        self.vocab_size = config.vocab_size
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.vit = HunYuanVisionTransformer(config.vision_config)
        self.config = config
        self.post_init()

    def set_decoder(self, decoder):
        self.model = decoder

    def get_decoder(self):
        return self.model

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.LongTensor] = None,
        past_key_values: Optional[Cache] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
 

Qua phân tích mã nguồn lớp `HunYuanVLForConditionalGeneration`, chúng ta nhận thấy sự không đồng nhất nghiêm trọng giữa hai luồng xử lý chính:
- `generate`: Đã được tích hợp sẵn bộ trích xuất đặc trưng hình ảnh `self.vit`, cơ chế chuyển đổi thiết bị `Device Mapping` và kỹ thuật chèn đặc trưng vào văn bản `masked_scatter`.
- `forward`: Hoàn toàn thiếu vắng các tham số liên quan đến hình ảnh `pixel_values`, `image_grid_thw` và không gọi đến bộ trích xuất `vit`.

Trong quá trình `SFT (Supervised Fine-Tuning)`, thư viện `Trainer` sẽ gọi hàm `forward`. Vì hàm này không thêm đặc trưng ảnh vào, mô hình chỉ nhận được các token văn bản và các token giữ chỗ `placeholder` rỗng.

Kết quả: Mô hình vẫn tính toán Loss dựa trên việc dự đoán chuỗi mục tiêu, nhưng nó chỉ dựa vào xác suất thống kê của ngôn ngữ

-> Loss giảm và mô hình bắt đầu xuất ra đúng định dạng, nhưng thực tế nó đang "học thuộc lòng" nội dung nhãn thay vì học từ ảnh.

# Edit `forward`

```python
def forward(
    self,
    input_ids: Optional[torch.LongTensor] = None,
    attention_mask: Optional[torch.Tensor] = None,
    position_ids: Optional[torch.LongTensor] = None,
    past_key_values: Optional[Cache] = None,
    pixel_values: Optional[torch.FloatTensor] = None,
    image_grid_thw: Optional[torch.FloatTensor] = None,
    inputs_embeds: Optional[torch.FloatTensor] = None,
    labels: Optional[torch.LongTensor] = None,
    use_cache: Optional[bool] = None,
    cache_position: Optional[torch.LongTensor] = None,
    logits_to_keep: Union[int, torch.Tensor] = 0,
    **kwargs: Unpack[TransformersKwargs],
) -> CausalLMOutputWithPast:
    r"""
    Example:

    ```python
    >>> from transformers import AutoProcessor, HunYuanVLForConditionalGeneration
    >>> from PIL import Image
    >>> import torch

    >>> model_name_or_path = "tencent/HunyuanOCR"
    >>> processor = AutoProcessor.from_pretrained(model_name_or_path, use_fast=False)
    >>> model = HunYuanVLForConditionalGeneration.from_pretrained(
    ...     model_name_or_path,
    ...     attn_implementation="eager",
    ...     torch_dtype=torch.bfloat16,
    ...     device_map="auto",
    ... )

    >>> img_path = "path/to/your/image.jpg"
    >>> image = Image.open(img_path).convert("RGB")

    >>> messages = [
    ...     {
    ...         "role": "user",
    ...         "content": [
    ...             {"type": "image", "image": img_path},
    ...             {"type": "text", "text": "Extract the text from the image."},
    ...         ],
    ...     }
    ... ]
    >>> text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    >>> inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to(model.device)

    >>> with torch.no_grad():
    ...     generated_ids = model.generate(**inputs, max_new_tokens=1024)
    >>> generated_ids_trimmed = generated_ids[0][len(inputs["input_ids"][0]):]
    >>> output = processor.decode(generated_ids_trimmed, skip_special_tokens=True)

    >>> print(output)

    ```"""
    if inputs_embeds is None:
        inputs_embeds = self.model.embed_tokens(input_ids).clone()
      
    if  pixel_values is not None:
        pixel_values = pixel_values.to(torch.bfloat16)
        image_embeds = self.vit(pixel_values, image_grid_thw)

        # ViT may be deployed on different GPUs from those used by LLMs, due to auto-mapping of accelerate.
        image_embeds = image_embeds.to(input_ids.device, non_blocking=True)

        image_mask, _ = self.get_placeholder_mask(
            input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds
        )
        inputs_embeds = inputs_embeds.masked_scatter(image_mask, image_embeds)
    
    outputs: BaseModelOutputWithPast = self.model(
        input_ids=None,
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=past_key_values,
        inputs_embeds=inputs_embeds,
        use_cache=use_cache,
        cache_position=cache_position,
        **kwargs,
    )

    hidden_states = outputs.last_hidden_state
    # Only compute necessary logits, and do not upcast them to float if we are not computing the loss
    slice_indices = slice(-logits_to_keep, None) if isinstance(logits_to_keep, int) else logits_to_keep
    logits = self.lm_head(hidden_states[:, slice_indices, :])

    loss = None
    if labels is not None:
        loss = self.loss_function(logits=logits, labels=labels, vocab_size=self.config.vocab_size, **kwargs)

    return CausalLMOutputWithPast(
        loss=loss,
        logits=logits,
        past_key_values=outputs.past_key_values,
        hidden_states=outputs.hidden_states,
        attentions=outputs.attentions,
    )
```

# Template chat

In [ ]:
from transformers import AutoProcessor
from PIL import Image

model_name_or_path = "tencent/HunyuanOCR"
processor = AutoProcessor.from_pretrained(model_name_or_path, use_fast=False)
img_path = "exmaple.jpg"
image_inputs = Image.open(img_path).convert("RGB")
PROMPT = "This is prompt"
ASSISTANT_ANSWER = "This is assistant answer"
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img_path},
            {"type": "text", "text": PROMPT},
        ],
    },
    {
        "role": "assistant",
        "content": ASSISTANT_ANSWER
    },
]
messages = [messages1]
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
    for msg in messages
]

print(texts[0])

<｜hy_begin▁of▁sentence｜><｜hy_place▁holder▁no▁100｜><｜hy_place▁holder▁no▁102｜><｜hy_place▁holder▁no▁101｜>This is prompt<｜hy_User｜>This is assistant answer<｜hy_Assistant｜>


Cấu trúc `template chat` của mô hình khác biệt so với các mô hình thông thường
- Mô hình thông thường: `<im_start>user This is prompt <im_start>assistant This is assistant answer`
- Mô hình `HunYuanOCR`: `This is prompt<｜hy_User｜>This is assistant answer<｜hy_Assistant｜>`

Vì vậy, cách lấy label cho `This is assistant answer`:

```pthon
...

user_id = tokenizer.convert_tokens_to_ids("<｜hy_User｜>")
assistant_id = tokenizer.convert_tokens_to_ids("<｜hy_Assistant｜>")

...

for i in range(input_ids.size(0)):
    ids = input_ids[i].tolist()

    if user_id in ids and assistant_id in ids:
        user_pos = ids.index(user_id)
        assistant_pos = len(ids) -1 - ids[::-1].index(assistant_id)
        labels[i, user_pos + 1: assistant_pos + 1] = input_ids[i, user_pos + 1: assistant_pos + 1]
```